### Machine Learning Pipeline

The dataset used for this example is obtained from [UC Irvine Machine Learning Repository](https://archive.ics.uci.edu/dataset/222/bank+marketing).

The second part consists of creating a machine learning model to predict if the client will subscribe (yes/no) a term deposit (variable y).

#### Import Dataset

In [6]:
!python -m pip install pandas

  Using cached tzdata-2026.2-py2.py3-none-any.whl.metadata (1.4 kB)
   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   ----- ---------------------------------- 1.3/9.8 MB 6.1 MB/s eta 0:00:02
   ------------ --------------------------- 3.1/9.8 MB 8.0 MB/s eta 0:00:01
   ---------------------- ----------------- 5.5/9.8 MB 9.1 MB/s eta 0:00:01
   ------------------------------ --------- 7.3/9.8 MB 8.7 MB/s eta 0:00:01
   -------------------------------------- - 9.4/9.8 MB 9.2 MB/s eta 0:00:01
   ---------------------------------------- 9.8/9.8 MB 9.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   ------- -------------------------------- 2.4/12.3 MB 10.3 MB/s eta 0:00:01
   --------------- ------------------------ 4.7/12.3 MB 11.4 MB/s eta 0:00:01
   ---------------------- ----------------- 7.1/12.3 MB 11.5 MB/s eta 0:00:01
   ------------------------------- -------- 9.7/12.3 MB 11.6 MB/s eta 0:00:01
   ----------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
!python -m pip show pandas

Name: pandas
Version: 3.0.3
Summary: Powerful data structures for data analysis, time series, and statistics
Home-page: https://pandas.pydata.org
Author: 
Author-email: The Pandas Development Team <pandas-dev@python.org>
License: BSD 3-Clause License

 Copyright (c) 2008-2011, AQR Capital Management, LLC, Lambda Foundry, Inc. and PyData Development Team
 All rights reserved.

 Copyright (c) 2011-2026, Open source contributors.

 Redistribution and use in source and binary forms, with or without
 modification, are permitted provided that the following conditions are met:

 * Redistributions of source code must retain the above copyright notice, this
   list of conditions and the following disclaimer.

 * Redistributions in binary form must reproduce the above copyright notice,
   this list of conditions and the following disclaimer in the documentation
   and/or other materials provided with the distribution.

 * Neither the name of the copyright holder nor the names of its
   contribut

In [9]:
!python -m pip install joblib scikit-learn

  Using cached scipy-1.17.1-cp312-cp312-win_amd64.whl.metadata (60 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   --------------- ------------------------ 3.1/8.0 MB 16.8 MB/s eta 0:00:01
   -------------------------------- ------- 6.6/8.0 MB 16.8 MB/s eta 0:00:01
   ---------------------------------------- 8.0/8.0 MB 14.2 MB/s eta 0:00:00
Using cached scipy-1.17.1-cp312-cp312-win_amd64.whl (36.5 MB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)



[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
# import libraries
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import f1_score

In [14]:
# read dataset
df = pd.read_csv(r"C:\Users\ASUS\Desktop\EADA\3º trimestre\ML Operations\Activities\mlops-and-system-design\session 2\bank-full_train_test.csv")

In [4]:
# read dataset
df = pd.read_csv("datasets/bank-full_train_test.csv")

NameError: name 'pd' is not defined

In [15]:
df.head()


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,29,housemaid,married,primary,no,0,yes,no,cellular,15,jul,317,3,-1,0,unknown,no
1,57,management,divorced,secondary,no,396,no,no,cellular,12,aug,108,2,-1,0,unknown,no
2,54,blue-collar,married,primary,no,714,no,no,cellular,5,may,225,1,-1,0,unknown,no
3,52,services,married,secondary,no,2072,no,no,cellular,10,jul,224,1,-1,0,unknown,no
4,34,management,married,tertiary,no,1778,no,no,cellular,4,nov,358,1,162,2,failure,yes


#### Transform Dataset

In [16]:
# balance dataset
def balance_dataset(df: pd.DataFrame) -> pd.DataFrame:
    # Separate the classes
    df_y0 = df[df['y'] == 'no']
    df_y1 = df[df['y'] == 'yes']

    # Find the smaller class size
    min_size = len(df_y1)

    # Randomly sample from each class
    df_y0_balanced = df_y0.sample(n=min_size, random_state=42)

    # Concatenate back together
    df_balanced = pd.concat([df_y0_balanced, df_y1])

    # Shuffle the dataset
    df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

    return df_balanced

df = balance_dataset(df)

In [18]:
import pandas as pd

In [20]:
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,43,blue-collar,married,primary,no,50,no,no,unknown,28,may,90,8,-1,0,unknown,no
1,32,technician,single,secondary,no,40,no,no,cellular,8,aug,419,2,-1,0,unknown,yes
2,32,self-employed,single,tertiary,no,2559,no,no,cellular,7,jul,889,1,-1,0,unknown,yes
3,23,student,single,secondary,no,480,no,no,cellular,9,feb,742,2,182,1,failure,yes
4,32,management,single,tertiary,no,1169,yes,no,unknown,21,may,322,2,-1,0,unknown,no


In [23]:
# map target
df.loc[df["y"] == "yes", "y"] = "1"
df.loc[df["y"] == "no", "y"] = "0"
df["y"] = df["y"].astype(int)

In [24]:
# prepare dataset for classification model
class Transformer:
    def __init__(self):
        self.DROP_COLUMNS = []
        self.BINARY_FEATURES = [
            "housing",
            "loan",
            "default",
        ]

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.drop(self.DROP_COLUMNS, axis=1)
        df = self._map_binary_column_to_int(df)
        df = self._map_month_to_int(df)

        return df

    def _map_binary_column_to_int(self, df: pd.DataFrame) -> pd.DataFrame:
        for col in self.BINARY_FEATURES:
            df[col] = df[col].map({"yes": 1, "no": 0})
        return df

    def _map_month_to_int(self, df: pd.DataFrame) -> pd.DataFrame:
        month_mapping = {
            "jan": 1,
            "feb": 2,
            "mar": 3,
            "apr": 4,
            "may": 5,
            "jun": 6,
            "jul": 7,
            "aug": 8,
            "sep": 9,
            "oct": 10,
            "nov": 11,
            "dec": 12,
        }
        df["month"] = df["month"].map(month_mapping)

        return df

In [25]:
# transform the dataset
df = Transformer().transform(df)

In [26]:
# encode categorical variables
ONE_HOT_ENCODE_COLUMNS = [
            "marital",
            "job",
            "education",
            "poutcome",
            "contact",
        ]

encoder = OneHotEncoder(drop='first', sparse_output=False).set_output(transform="pandas")
encoder.fit(df[ONE_HOT_ENCODE_COLUMNS])
encoded_df = encoder.transform(df[ONE_HOT_ENCODE_COLUMNS])
df = df.drop(columns=ONE_HOT_ENCODE_COLUMNS)
df = pd.concat([df, encoded_df], axis=1)

In [28]:
import os

# Crear la carpeta si no existe
os.makedirs("model_utils", exist_ok=True)

# Ahora guardar el encoder
joblib.dump(encoder, 'model_utils/encoder.pkl')

['model_utils/encoder.pkl']

In [29]:
# save encoder - future use
joblib.dump(encoder, 'model_utils/encoder.pkl')

['model_utils/encoder.pkl']

#### Train and evaluate model

In [30]:
# Split the data into training and test sets
X = df.drop('y', axis=1)
y = df['y']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [33]:
# Define the model hyperparameters
params = {
    "solver": "lbfgs",
    "max_iter": 1000,

    "random_state": 8888,
}

In [34]:
# Train the model
lr = LogisticRegression(**params)
lr.fit(X_train, y_train)

c:\Users\ASUS\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",8888
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [35]:
# Predict on the test set
y_pred = lr.predict(X_test)

In [36]:
# Calculate metrics
f1_score = f1_score(y_test, y_pred)

### MLFlow Integration

In [38]:
!python -m pip install mlflow

  Using cached flask-3.1.3-py3-none-any.whl.metadata (3.2 kB)
  Using cached alembic-1.18.4-py3-none-any.whl.metadata (7.2 kB)
  Using cached cryptography-46.0.7-cp311-abi3-win_amd64.whl.metadata (5.7 kB)
  Using cached docker-7.1.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached graphene-3.4.3-py2.py3-none-any.whl.metadata (6.9 kB)
  Using cached matplotlib-3.10.9-cp312-cp312-win_amd64.whl.metadata (52 kB)
  Using cached sqlalchemy-2.0.49-cp312-cp312-win_amd64.whl.metadata (9.8 kB)
  Using cached waitress-3.0.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached click-8.3.3-py3-none-any.whl.metadata (2.6 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
  Using cached databricks_sdk-0.108.0-py3-none-any.whl.metadata (43 kB)
  Using cached fastapi-0.136.1-py3-none-any.whl.metadata (28 kB)
  Using cached gitpython-3.1.50-py3-none-any.whl.metadata (14 kB)
  Using cached opentelemetry_api-1.41.1-py3-none-any.whl.metadata (1.5 kB)
  Using cached opentelemetry_sdk-1.41

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  You can safely remove it manually.
  Consider adding this directory to

In [39]:
# import libraries
import mlflow
from mlflow.models import infer_signature

In a new terminal, run a local Tracking Server with the following command:

`mlflow server --host 127.0.0.1 --port 8080`

In [40]:
# Set our tracking server uri for logging
mlflow.set_tracking_uri(uri="http://127.0.0.1:8080")

In [41]:
# Create a new MLflow Experiment
mlflow.set_experiment("LR Experiment")

# Start an MLflow run
with mlflow.start_run():
    # Log the hyperparameters
    mlflow.log_params(params)

    # Log the loss metric
    mlflow.log_metric("f1_score", f1_score)

    # Set a tag that we can use to remind ourselves what this run was for
    mlflow.set_tag("Training Info", "Basic LR model to present MLFlow capabilities")

    # Infer the model signature
    signature = infer_signature(X_train, lr.predict(X_train))

    # Log the model
    model_info = mlflow.sklearn.log_model(
        sk_model=lr,
        artifact_path="bank_model",
        signature=signature,
        input_example=X_train,
        registered_model_name="bank-model-basic",
    )

2026/05/14 19:35:41 INFO mlflow.tracking.fluent: Experiment with name 'LR Experiment' does not exist. Creating a new experiment.
c:\Users\ASUS\AppData\Local\Programs\Python\Python312\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/05/14 19:35:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please 

🏃 View run adaptable-chimp-212 at: http://127.0.0.1:8080/#/experiments/564433805273450941/runs/6961ed15237247159cf1dc309bd42aed
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/564433805273450941


MlflowException: API request to endpoint /api/2.0/mlflow/logged-models failed with error code 404 != 200. Response body: '<!doctype html>
<html lang=en>
<title>404 Not Found</title>
<h1>Not Found</h1>
<p>The requested URL was not found on the server. If you entered the URL manually please check your spelling and try again.</p>
'